# Fraud Detection Model



# Cloud-Based Fraud Detection Analytics on AWS

This notebook demonstrates a simple fraud detection workflow using Amazon SageMaker.

## Workflow
1. Load transaction data from Amazon S3
2. Explore and preprocess the dataset
3. Engineer time-based features from timestamp
4. Train a Random Forest fraud detection model
5. Evaluate model performance
6. Run a sample fraud prediction
7. Save the trained model back to S3

In [14]:
import pandas as pd
import numpy as np
import boto3
import joblib
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# =========================
# 1. Load data from S3
# =========================
bucket_name = "finalproject-fraud-detection/raw"
file_key = "fraud_transactions.csv"

s3_path = f"s3://{bucket_name}/{file_key}"
df = pd.read_csv(s3_path)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
df.head()

ModuleNotFoundError: No module named 'boto3'

In [ ]:
# =========================
# 2. Explore data
# =========================
print("Column names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

print("\nFraud label distribution:")
print(df["fraud"].value_counts(dropna=False))

# =========================
# 3. Time feature engineering
# =========================
# Convert timestamp into usable time-based features
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Create new features from timestamp
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek   # Monday=0, Sunday=6
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_night"] = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)

# Drop original timestamp after extracting features
df = df.drop(columns=["timestamp"])

print("\nDataset after time feature engineering:")
df.head()

print("\nSummary statistics:")
df.describe(include="all")

In [ ]:
# =========================
# 4. Preprocessing
# =========================
df_model = df.copy()

# Remove rows with missing fraud label if any
df_model = df_model.dropna(subset=["fraud"])

# Common non-feature ID columns
drop_if_exists = ["transaction_id"]
for col in drop_if_exists:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# Encode categorical columns automatically
label_encoders = {}
categorical_cols = df_model.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# Fill any remaining missing values
for col in df_model.columns:
    if df_model[col].dtype in ["float64", "int64"]:
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Define features and target
X = df_model.drop(columns=["fraud"])
y = df_model["fraud"]

print("Final feature columns:")
print(X.columns.tolist())

# =========================
# 5. Train/test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 6. Train model
# =========================
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("\nModel training completed.")
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

In [ ]:
# =========================
# 7. Evaluate model
# =========================
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Model Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature importance for interpretation
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop Feature Importances:")
feature_importance

In [ ]:
# =========================
# 8. Single transaction prediction demo
# =========================
sample_input = {}

# Create a sample transaction using the same feature columns
for col in X.columns:
    if col == "amount":
        sample_input[col] = 1200.0
    elif col == "hour":
        sample_input[col] = 2
    elif col == "day_of_week":
        sample_input[col] = 6
    elif col == "is_weekend":
        sample_input[col] = 1
    elif col == "is_night":
        sample_input[col] = 1
    else:
        # for encoded categorical or other numeric columns,
        # use the most frequent value from training data as default
        sample_input[col] = X[col].mode()[0]

sample_df = pd.DataFrame([sample_input])

prediction = model.predict(sample_df)[0]
prediction_proba = model.predict_proba(sample_df)[0]

print("Sample transaction used for prediction:")
sample_df

print("Prediction result:", prediction)
print("Prediction probabilities:", prediction_proba)

if prediction == 1:
    print("Interpretation: This transaction is predicted as FRAUD.")
else:
    print("Interpretation: This transaction is predicted as NORMAL.")

In [9]:
# =========================
# 9. Save artifacts
# =========================
joblib.dump(model, "fraud_model.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")

print("Artifacts saved locally:")
print("- fraud_model.pkl")
print("- label_encoders.pkl")
print("- feature_columns.pkl")

# =========================
# 10. Upload artifacts to S3
# =========================
s3_client = boto3.client("s3")

actual_bucket_name = "finalproject-fraud-detection"
model_prefix = "model"

s3_client.upload_file("fraud_model.pkl", bucket_name, "model/fraud_model.pkl")
s3_client.upload_file("label_encoders.pkl", bucket_name, "model/label_encoders.pkl")
s3_client.upload_file("feature_columns.pkl", bucket_name, "model/feature_columns.pkl")

print("\nArtifacts uploaded to S3 successfully:")
print(f"s3://{bucket_name}/model/fraud_model.pkl")
print(f"s3://{bucket_name}/model/label_encoders.pkl")
print(f"s3://{bucket_name}/model/feature_columns.pkl")

NameError: name 'joblib' is not defined

KeyError: 'spark'